In [ ]:
import pandas as pd
import os
from highstreets import config
from highstreets.data_source_sink.dataloader import DataLoader
from highstreets.data_source_sink.datawriter import DataWriter
from highstreets.data_transformation.mcard_transform import McardTransform
from highstreets.core.sql_manager import SQLManager
from highstreets.data_transformation.mcard_weekly_processor import FileProcessor
from highstreets.api.clientbase import APIClient
from sqlalchemy import create_engine
import psycopg2
from dotenv import find_dotenv, load_dotenv
load_dotenv(find_dotenv())

base_dir = config.BASE_DIR
# initialize the database connection
database = os.getenv("PG_DATABASE")
username = os.getenv("PG_USER")
password = os.getenv("PG_PASSWORD")
host = os.getenv("PG_HOST")
port = os.getenv("PG_PORT")
engine = create_engine(
    f"postgresql+psycopg2://{username}:{password}@" f"{host}:{port}/{database}"
)

# instantiate the classes
data_loader = DataLoader()
data_writer = DataWriter()
mcard_transform = McardTransform()
sql_manager = SQLManager()
api_client = APIClient()
dir_path = f"{base_dir}mastercard/sharefile_test"
mcard_weekly = FileProcessor(data_loader, data_writer, dir_path)

# Connect to PostgreSQL database
conn = psycopg2.connect(
    dbname=os.getenv("PG_DATABASE"),
    user=os.getenv("PG_USER"),
    password=os.getenv("PG_PASSWORD"),
    host=os.getenv("PG_HOST"),
    port=os.getenv("PG_PORT"),
)

## Test adjusted 3-hourly spend

In [ ]:
adj_factor = data_loader.get_full_data('econ_busyness_mcard_adjustment_factors')
cpi_table = data_loader.get_full_data('econ_busyness_mcard_cpi_data')


In [ ]:


def clean_3hourly_quad_data(quad_3hourly,fill_io_nas=True):
    quad_3hourly['yr']=quad_3hourly['count_date'].dt.year
    quad_3hourly['month']=quad_3hourly['count_date'].dt.month


    #Load inner outer quad lookup
    io_lookup = data_loader.get_full_data("econ_busyness_mcard_inner_outer_quad_lookup")
    io_lookup['quad_id'] = io_lookup['quad_id'].astype('Int64')
    io_lookup = io_lookup.sort_values(by=['inner_outer']).drop_duplicates(subset=['quad_id'], keep='last')
    #io_lookup = io_lookup.drop_duplicates(subset=['quad_id'])
    #io_lookup['quad_id'] = np.where(io_lookup['quad_id'].isin(diff_quads),'Inner',io_lookup['quad_id'])

    quad_3hourly['quad_id'] = quad_3hourly['quad_id'].astype('Int64')

    merged = pd.merge(quad_3hourly,io_lookup,on='quad_id',how='left')

    if fill_io_nas:
        unassigned_quads = merged[merged['inner_outer'].isnull()]['quad_id'].unique()
        merged['inner_outer'] = merged['inner_outer'].fillna('Outer')


    merged_af = pd.merge(merged,adj_factor,how='left',on=['inner_outer','yr','month'])

    # Adjust
    merged_af['txn_amt_adj_new'] = merged_af['txn_amt']/merged_af['adjustment_factor_retail']

    merged_af = mcard_transform.inflation_adjust(
    merged_af,
    cpi_table[cpi_table['aggregate']=='Overall Index'],
    reindexing_year=2018,
    col_to_adjust=["txn_amt_adj_new"],
    date_col="count_date",
)
    
    merged_af['txn_amt_adj_new'] = merged_af['txn_amt_adj_new'].round(4)
    
    if fill_io_nas:
        merged_af = merged_af[~(merged_af['quad_id'].isin(unassigned_quads))]
        
    merged_af_io = merged_af[merged_af['hours']=='12-15'].sort_values(by='count_date').groupby(['count_date','inner_outer'])[['txn_amt','txn_amt_adj','txn_amt_adj_new']].sum(min_count=1).reset_index()
    ldn = merged_af[merged_af['hours']=='12-15'].sort_values(by='count_date').groupby(['count_date'])[['txn_amt','txn_amt_adj','txn_amt_adj_new']].sum(min_count=1).reset_index()

    return merged_af_io, ldn, merged_af


In [ ]:
# quad_3hourly_new = data_loader.get_full_data('test_econ_busyness_mrli_3hourly_adj')


In [ ]:
merged_af_io_new,ldn,quad_adj = clean_3hourly_quad_data(quad_3hourly_new, fill_io_nas=True)
merged_af_io_new

### Plot quad-level 3-hourly aggregated to Inner/Outer LDN

In [ ]:
import matplotlib.pyplot as plt

## Plot these differences between txn_amt_adj and txn_amt_adj_new 
for io in ['Inner','Outer']:
    #df = merged_af_io[merged_af_io['inner_outer']==io]
    df_new = merged_af_io_new[merged_af_io_new['inner_outer']==io]

    #df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
    plt.figure(figsize=(14,4))
    # This one is currently wrong:
    #plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
    #plt.plot(df['count_date'],df['txn_amt_adj_new'],label='Correct version - adjusted in Python')
    plt.plot(df_new['count_date'],df_new['txn_amt'],label='txn_amt')
    plt.plot(df_new['count_date'],df_new['txn_amt_adj_new'],label='txn_amt adjusted w Python code')
    plt.plot(df_new['count_date'],df_new['txn_amt_adj'],label='New txn_amt_adj - adjusted w SQL code')

    plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
    plt.ylim(ymin=0)
    #plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
    plt.title(io+" London: 3-hourly summed")
    plt.show()

## Plot and load aggregated areas

In [ ]:
def agg_quad_to_area(quad_df, quad_area_lookup, area_ids = []):
    quad_df['quad_id'] = quad_df['quad_id'].astype('Int64')

    area_lookup = data_loader.get_full_data(quad_area_lookup)
    area_lookup['quad_id'] = area_lookup['quad_id'].astype('Int64')

    area_df = pd.merge(quad_df,area_lookup,how='inner',on='quad_id')

    area_df = (area_df.groupby(
        area_ids+['count_date','hours'])[
            'txn_amt','txn_amt_adj','txn_amt_adj_new']
            .sum(min_count=1).
            reset_index())



    return area_df

In [ ]:
hs = data_loader.get_full_data('aws_econ_busyness_mcard_highstreets_3hourly_txn')
tc = data_loader.get_full_data('aws_econ_busyness_mcard_towncentres_3hourly_txn')
bid = data_loader.get_full_data('aws_econ_busyness_mcard_bids_3hourly_txn')
bespoke = data_loader.get_full_data('aws_econ_busyness_mcard_bespokes_3hourly_txn')



In [ ]:

# new_hs = agg_quad_to_area(quad_adj, 
#                  'econ_busyness_mcard_highstreets_quad_lookup', 
#                  area_ids = ['highstreet_id','highstreet_name'])

# new_tc = agg_quad_to_area(quad_adj, 
#                  'econ_busyness_mcard_TownCentres_quad_lookup', 
#                  area_ids = ['tc_id','tc_name'])

# new_bid = agg_quad_to_area(quad_adj, 
#                  'econ_busyness_mcard_BIDs_quad_lookup', 
#                  area_ids = ['bid_id','bid_name'])

# new_bespoke = agg_quad_to_area(quad_adj, 
#                  'econ_busyness_mcard_bespoke_quad_lookup', 
#                  area_ids = ['bespoke_area_id','name'])

In [ ]:
dfs = {'hs':[hs,'highstreet_id'],
       'tc':[tc,'tc_id'],
       'bid':[bid,'bid_id'],
       'bespoke':[bespoke,'bespoke_area_id']}

In [ ]:
for name in dfs.keys():
    dfs[name][0].to_csv(f"Z:/HSDS/data/mastercard/mrli_3hourly/test/mcard_test_{name}.csv",index=False)

In [ ]:
## Plot these differences between txn_amt_adj and txn_amt_adj_new 
import matplotlib.pyplot as plt

for name in dfs.keys():
    #df = merged_af_io[merged_af_io['inner_outer']==io]
    area = dfs[name][0][dfs[name][0][dfs[name][1]]==1]
    area = area[area['hours']=="12-15"]
    #area_new = dfs[name][-1][dfs[name][-1][dfs[name][1]]==1]
    #df1 = adj_factor_new[adj_factor_new['inner_outer']==io]
    plt.figure(figsize=(14,4))
    # This one is currently wrong:
    #plt.plot(df['count_date'],df['txn_amt_adj'],label='txn_amt_adj') 
    #plt.plot(area_new['count_date'],area_new['txn_amt_adj'],label='Aggregated quad values - adjusted')
    plt.plot(area['count_date'],area['txn_amt'],label='txn_amt')
    plt.plot(area['count_date'],area['txn_amt_adj'],label='New txn_amt_adj - adjusted w SQL code')

    plt.legend()   #plt.plot(df1['adjustment_factor_retail'])
    plt.ylim(ymin=0)
    #plt.xlim(xmin=np.datetime64('2020-10-01'),xmax=np.datetime64('2021-10-01'))
    plt.title(f"{name}")
    plt.show()

    